# 选修E3 · Day 2 上机：LLM 应用工程

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

**核心命题**：用 RAG 让 LLM 基于产品知识库生成营销文案，Prompt 工程控制输出，LangSmith 追踪全链路，RAGAS 评估质量。

**真实库**：tiktoken（token 计数）+ langchain_core（Prompt 模板）+ langsmith（追踪）+ numpy（RAG 检索）


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> ⚠️ 全部使用本地库，无需 OPENAI_API_KEY。无 API key 时用 mock LLM + Prompt 模板演示。
> tiktoken / langchain-core / langsmith / numpy 均为纯本地库，秒级加载。


In [ ]:
# !pip install tiktoken langchain-core langsmith numpy -q

import tiktoken
import numpy as np
import math
import json
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

print("=== 环境就绪 ===")
print(f"tiktoken: {tiktoken.__version__}")
print(f"numpy: {np.__version__}")
print(f"langchain_core: 可用")
print(f"langsmith: 可用")


## 1. 场景背景与营销映射

**核心命题**：LLM 应用工程是营销 Agent 的"应用层"。本上机解决一个完整场景：
- 用 **tiktoken** 计算营销文案 token 数 + 推理成本（gpt-4o vs DeepSeek V3）
- 用 **ChatPromptTemplate** 构建营销文案生成 Prompt
- 用 **@traceable** 追踪 LLM 调用
- 用 **numpy TF-IDF** 实现 RAG 检索（营销知识库）
- 用 **RAG + Prompt + mock LLM** 生成基于知识库的营销文案
- 用 **RAGAS 简化实现** 评估 RAG 质量

**营销知识库**（智能手表产品文档，5 个文档）：


In [ ]:
# 营销知识库：智能手表产品文档（基于真实产品文档结构构建，见 data/README.md）
knowledge_base = [
    {"id": "doc1", "content": "智能手表Pro支持7天超长续航，采用低功耗芯片2.0，日常使用可达7天，运动模式3天。"},
    {"id": "doc2", "content": "手表内置100+运动模式，包括跑步、游泳、骑行、瑜伽，支持自动识别6种运动。"},
    {"id": "doc3", "content": "心率血氧监测功能，24小时连续心率监测，血氧饱和度SpO2测量，异常心率提醒。"},
    {"id": "doc4", "content": "手表防水等级5ATM，支持游泳佩戴，50米防水深度。"},
    {"id": "doc5", "content": "品牌故事：致力于用科技守护健康，让每个人都能享受智能穿戴带来的便利。"},
]

print(f"知识库文档数: {len(knowledge_base)}")
for doc in knowledge_base:
    print(f"  {doc['id']}: {doc['content'][:40]}...")


## TODO 1：用 tiktoken 计算营销文案 token 数 + 推理成本

**tiktoken** 是 OpenAI 的 BPE 分词器，精确计算 token 数：
- `o200k_base`：gpt-4o 的编码
- `cl100k_base`：gpt-4/3.5 和 DeepSeek V3 的编码

**任务**：
1. 用 `tiktoken.get_encoding('o200k_base')` 和 `cl100k_base` 加载编码器
2. 对中英文营销文案做 tokenization，对比 token 消耗
3. 结合 gpt-4o（$2.50/$10.00 per M）和 DeepSeek V3（$0.27/$1.10 per M）定价，计算日均 1000 次调用的月成本


In [ ]:
# 用 tiktoken 对营销文案做 tokenization + 推理成本计算
enc_o200k = tiktoken.get_encoding('o200k_base')   # gpt-4o
enc_cl100k = tiktoken.get_encoding('cl100k_base')  # DeepSeek V3 / gpt-4

marketing_copy_zh = "这款智能手表续航长达7天，支持100+运动模式，心率血氧实时监测，让你的健康尽在掌握。"
marketing_copy_en = "This smartwatch lasts 7 days on a single charge, supports 100+ sports modes, with real-time heart rate and SpO2 monitoring."

tokens_zh_o200k = enc_o200k.encode(marketing_copy_zh)
tokens_en_o200k = enc_o200k.encode(marketing_copy_en)
tokens_zh_cl100k = enc_cl100k.encode(marketing_copy_zh)
tokens_en_cl100k = enc_cl100k.encode(marketing_copy_en)

print("=== Token 计数对比 ===")
print(f"中文文案 ({len(marketing_copy_zh)} chars):")
print(f"  o200k_base (gpt-4o):   {len(tokens_zh_o200k)} tokens")
print(f"  cl100k_base (DS V3):   {len(tokens_zh_cl100k)} tokens")
print(f"英文文案 ({len(marketing_copy_en)} chars):")
print(f"  o200k_base (gpt-4o):   {len(tokens_en_o200k)} tokens")
print(f"  cl100k_base (DS V3):   {len(tokens_en_cl100k)} tokens")
print(f"中文/英文 token 比 (o200k): {len(tokens_zh_o200k)/len(tokens_en_o200k):.2f}x")

# 推理成本计算
pricing = {
    "gpt-4o":      {"input": 2.50, "output": 10.00},
    "DeepSeek V3": {"input": 0.27, "output": 1.10},
}
daily_calls = 1000
avg_input_tokens = 500
avg_output_tokens = 200
days_per_month = 30

print("\n=== 推理成本对比（日均 1000 次调用）===")
print(f"假设：input {avg_input_tokens} tokens/次, output {avg_output_tokens} tokens/次")
for model, p in pricing.items():
    monthly_input = avg_input_tokens * daily_calls * days_per_month
    monthly_output = avg_output_tokens * daily_calls * days_per_month
    cost = (monthly_input / 1_000_000 * p["input"] +
            monthly_output / 1_000_000 * p["output"])
    print(f"  {model:14s}: input ${p['input']:.2f}/M, output ${p['output']:.2f}/M -> ${cost:.2f}/月")

gpt4o_cost = (avg_input_tokens * daily_calls * days_per_month / 1e6 * 2.50 +
              avg_output_tokens * daily_calls * days_per_month / 1e6 * 10.00)
ds_cost = (avg_input_tokens * daily_calls * days_per_month / 1e6 * 0.27 +
           avg_output_tokens * daily_calls * days_per_month / 1e6 * 1.10)
print(f"\n  DeepSeek V3 仅为 gpt-4o 成本的 {ds_cost/gpt4o_cost*100:.1f}%")


## TODO 2：用 ChatPromptTemplate 构建营销文案生成 Prompt

**ChatPromptTemplate**（langchain_core）构建结构化 Prompt：
- `from_messages([("system", ...), ("human", ...)])` 定义 System + Human 消息
- `format_messages(...)` 填充变量生成消息列表

**任务**：
1. 用 ChatPromptTemplate.from_messages 构建 Prompt（system 定义角色+约束，human 定义产品信息）
2. 用 format_messages 填充 product_name / selling_points / target_audience
3. 打印格式化后的消息列表


In [ ]:
# 用 ChatPromptTemplate 构建营销文案生成 Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个营销文案专家。请根据产品信息生成吸引人的营销文案。要求：1) 突出核心卖点 2) 包含CTA 3) 控制在100字以内"),
    ("human", "产品名称：{product_name}\n核心卖点：{selling_points}\n目标受众：{target_audience}\n请生成营销文案。"),
])

# 格式化
formatted = prompt.format_messages(
    product_name="智能手表Pro",
    selling_points="7天续航, 100+运动模式, 心率血氧监测",
    target_audience="25-35岁都市白领"
)

print("=== ChatPromptTemplate 格式化结果 ===")
for msg in formatted:
    print(f"[{msg.type}]: {msg.content}")
    print()

# StrOutputParser 演示
parser = StrOutputParser()
print(f"StrOutputParser 类型: {type(parser).__name__}")
print(f"Prompt 变量: {prompt.input_variables}")


## TODO 3：用 @traceable 追踪 LLM 调用（mock LLM）

**langsmith @traceable** 装饰器记录函数调用全链路。无 LANGSMITH_API_KEY 时仍可运行（本地模式）。

**任务**：
1. 定义 `@traceable(name="marketing_copy_generator")` 装饰的函数
2. 函数内部用 ChatPromptTemplate 构建 Prompt，用 mock LLM（模板生成）输出营销文案
3. 调用函数并打印结果


In [ ]:
# 用 @traceable 追踪 LLM 调用（mock LLM，无 API key）
@traceable(name="marketing_copy_generator")
def generate_marketing_copy(product_name, selling_points, target_audience):
    # Mock LLM 生成营销文案（无 API key 时用模板演示）
    prompt = ChatPromptTemplate.from_messages([
        ("system", "你是营销文案专家。根据产品信息生成100字以内营销文案。"),
        ("human", "产品：{product}\n卖点：{points}\n受众：{audience}\n生成营销文案。"),
    ])
    messages = prompt.format_messages(
        product=product_name, points=selling_points, audience=target_audience
    )
    # Mock LLM: 基于模板生成
    copy = f"【{product_name}】{selling_points}，专为{target_audience}打造。立即抢购，享受健康生活！"
    return {"copy": copy, "prompt_messages": len(messages), "tokens_estimated": len(copy)}

# 调用并追踪
t0 = time.time()
result = generate_marketing_copy("智能手表Pro", "7天续航+100种运动+心率血氧", "都市白领")
elapsed = time.time() - t0

print("=== @traceable 追踪营销文案生成 ===")
print(f"生成文案: {result['copy']}")
print(f"Prompt 消息数: {result['prompt_messages']}")
print(f"估算 token 数: {result['tokens_estimated']}")
print(f"耗时: {elapsed*1000:.1f}ms")
print(f"追踪名称: marketing_copy_generator（LangSmith @traceable 已装饰）")


## TODO 4：用 numpy TF-IDF + 余弦相似度实现 RAG 检索

**RAG 检索**：从营销知识库中召回与用户问题最相关的文档。
- **TF-IDF**：词频-逆文档频率，将文档转为向量
- **余弦相似度**：衡量 query 与 doc 的相似度

**任务**：
1. 实现 `tfidf_vectorize(docs, query)` 函数：构建词表 -> 计算 TF-IDF -> 余弦相似度
2. 对 query="手表续航多久" 检索，打印 top-3 召回结果
3. （可选）对比 sentence-transformers all-MiniLM-L6-v2 的检索效果


In [ ]:
# 用 numpy TF-IDF + 余弦相似度实现 RAG 检索
def tfidf_vectorize(docs, query):
    # TF-IDF 向量化 + 余弦相似度检索
    # 构建词表（字符级，适合中文）
    vocab = set()
    for d in docs:
        vocab.update(d['content'])
    vocab.update(query)
    vocab = sorted(vocab)
    vocab_idx = {w: i for i, w in enumerate(vocab)}

    # TF（词频/文档长度）
    def tf(text):
        v = np.zeros(len(vocab))
        for w in text:
            v[vocab_idx[w]] += 1
        return v / max(len(text), 1)

    # IDF
    N = len(docs)
    idf = np.zeros(len(vocab))
    for w, i in vocab_idx.items():
        df = sum(1 for d in docs if w in d['content'])
        idf[i] = math.log((N + 1) / (df + 1)) + 1

    # TF-IDF
    doc_vectors = np.array([tf(d['content']) * idf for d in docs])
    query_vector = tf(query) * idf

    # 余弦相似度
    def cosine(a, b):
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        return float(np.dot(a, b) / (na * nb + 1e-10))

    scores = [cosine(query_vector, dv) for dv in doc_vectors]
    return scores

# 检索
query = "手表续航多久"
scores = tfidf_vectorize(knowledge_base, query)
ranked = sorted(zip(knowledge_base, scores), key=lambda x: -x[1])

print("=== RAG 检索结果（numpy TF-IDF + 余弦相似度）===")
print(f"Query: {query}")
print(f"\nTop-3 召回:")
for rank, (doc, score) in enumerate(ranked[:3], 1):
    print(f"  {rank}. [{score:.4f}] {doc['id']}: {doc['content'][:50]}...")

# 对比另一个 query
query2 = "游泳能戴吗"
scores2 = tfidf_vectorize(knowledge_base, query2)
ranked2 = sorted(zip(knowledge_base, scores2), key=lambda x: -x[1])
print(f"\nQuery: {query2}")
print(f"Top-3 召回:")
for rank, (doc, score) in enumerate(ranked2[:3], 1):
    print(f"  {rank}. [{score:.4f}] {doc['id']}: {doc['content'][:50]}...")

print(f"\n注：生产环境可用 sentence-transformers all-MiniLM-L6-v2 替代 TF-IDF，语义检索质量更高")


## TODO 5：用 RAG + Prompt + mock LLM 生成基于知识库的营销文案

**RAG 生成**：检索 -> Prompt 模板 -> LLM 生成

**任务**：
1. 定义 `@traceable(name="rag_marketing_qa")` 装饰的函数
2. 函数内部：检索 top-3 -> ChatPromptTemplate 构建 RAG Prompt -> mock LLM 生成回答
3. 调用函数问"智能手表的续航时间是多少？"，打印回答 + 检索来源


In [ ]:
# 用 RAG + Prompt + mock LLM 生成基于知识库的营销文案
@traceable(name="rag_marketing_qa")
def rag_marketing_qa(question, knowledge_base, top_k=3):
    # RAG 问答：检索 + Prompt + mock LLM 生成
    # Step 1: 检索
    scores = tfidf_vectorize(knowledge_base, question)
    ranked = sorted(zip(knowledge_base, scores), key=lambda x: -x[1])
    retrieved = ranked[:top_k]
    context = "\n".join([d['content'] for d, s in retrieved])

    # Step 2: Prompt 模板（RAG 专用）
    prompt = ChatPromptTemplate.from_messages([
        ("system", "你是营销知识助手。请基于以下上下文回答问题。如果上下文中没有相关信息，请说'根据现有资料无法回答'。不要编造信息。"),
        ("human", "上下文：\n{context}\n\n问题：{question}\n\n请基于上下文回答："),
    ])
    messages = prompt.format_messages(context=context, question=question)

    # Step 3: Mock LLM 生成（基于检索结果）
    top_doc = retrieved[0][0]['content'] if retrieved else "无相关资料"
    answer = f"根据产品资料：{top_doc[:80]}"

    return {
        "question": question,
        "answer": answer,
        "retrieved_docs": [{"id": d['id'], "score": float(s), "content": d['content']} for d, s in retrieved],
        "context_used": context[:200],
        "prompt_messages": len(messages),
    }

# 调用 RAG 问答
result = rag_marketing_qa("智能手表的续航时间是多少？", knowledge_base)

print("=== RAG 营销问答（@traceable 追踪）===")
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print(f"\n检索到 {len(result['retrieved_docs'])} 个文档:")
for doc in result['retrieved_docs']:
    print(f"  [{doc['score']:.4f}] {doc['id']}: {doc['content'][:50]}...")
print(f"\nPrompt 消息数: {result['prompt_messages']}")
print(f"使用的上下文（前 100 字）: {result['context_used'][:100]}...")

# 第二个问题
result2 = rag_marketing_qa("手表能戴着游泳吗？", knowledge_base)
print(f"\nQ: {result2['question']}")
print(f"A: {result2['answer']}")


## TODO 6：用 RAGAS 简化实现评估 RAG 质量

**RAGAS**（Retrieval-Augmented Generation Assessment）核心指标：
- **Faithfulness（忠实度）**：回答信息是否能在检索上下文中找到（防幻觉）
- **Context Recall（上下文召回率）**：ground truth 信息是否被检索到
- **Answer Relevance（回答相关性）**：回答是否切题

**任务**：
1. 实现 `ragas_simplified(question, answer, retrieved_docs, ground_truth)` 函数
2. 用字符重叠规则近似 faithfulness / context_recall / answer_relevance
3. 评估 TODO5 的 RAG 结果


In [ ]:
# 用 RAGAS 简化实现评估 RAG 质量（规则近似）
def ragas_simplified(question, answer, retrieved_docs, ground_truth=None):
    # RAGAS 简化评估：用字符重叠规则近似三个核心指标
    # 检索上下文
    context = " ".join([d['content'] for d in retrieved_docs])

    # Faithfulness: answer 中的信息是否能在 context 中找到（防幻觉）
    answer_chars = set(answer)
    context_chars = set(context)
    faithful_chars = answer_chars & context_chars
    faithfulness = len(faithful_chars) / max(len(answer_chars), 1)

    # Context Recall: ground truth 中的信息是否被检索到
    if ground_truth:
        gt_chars = set(ground_truth)
        recalled = gt_chars & context_chars
        context_recall = len(recalled) / max(len(gt_chars), 1)
    else:
        context_recall = 1.0  # 无 ground truth 时跳过

    # Answer Relevance: answer 是否包含 question 的关键词
    q_chars = set(question)
    relevance = len(q_chars & answer_chars) / max(len(q_chars), 1)

    return {
        "faithfulness": round(faithfulness, 4),
        "context_recall": round(context_recall, 4),
        "answer_relevance": round(relevance, 4),
    }

# 评估 TODO5 的 RAG 结果
question = result['question']
answer = result['answer']
retrieved = result['retrieved_docs']
ground_truth = "智能手表Pro支持7天超长续航"

scores = ragas_simplified(question, answer, retrieved, ground_truth)

print("=== RAGAS 简化评估（规则近似）===")
print(f"Question: {question}")
print(f"Answer: {answer}")
print(f"Ground Truth: {ground_truth}")
print(f"\n评估指标:")
print(f"  Faithfulness（忠实度）:     {scores['faithfulness']:.4f}  (answer 信息在 context 中的比例，防幻觉)")
print(f"  Context Recall（上下文召回）: {scores['context_recall']:.4f}  (ground truth 被检索到的比例)")
print(f"  Answer Relevance（回答相关性）: {scores['answer_relevance']:.4f}  (answer 与 question 的相关度)")
print(f"\n注：生产环境用 RAGAS 库 + LLM-as-Judge 做精确评估（见 reading.md RAGAS 论文 arXiv 2309.15217）")

# 评估第二个问题
scores2 = ragas_simplified(result2['question'], result2['answer'], result2['retrieved_docs'],
                            ground_truth="手表防水等级5ATM支持游泳")
print(f"\n第二个问题评估:")
print(f"  Faithfulness: {scores2['faithfulness']:.4f}")
print(f"  Context Recall: {scores2['context_recall']:.4f}")
print(f"  Answer Relevance: {scores2['answer_relevance']:.4f}")


## 总结

**本上机完成了 LLM 应用工程的全链路**：

| TODO | 任务 | 真实库 | 营销场景 |
|------|------|--------|---------|
| TODO1 | Token 计数 + 推理成本 | tiktoken | gpt-4o vs DeepSeek V3 定价对比 |
| TODO2 | Prompt 模板 | langchain_core ChatPromptTemplate | 营销文案生成 Prompt |
| TODO3 | LLM 追踪 | langsmith @traceable | 营销文案生成全链路追踪 |
| TODO4 | RAG 检索 | numpy TF-IDF + 余弦相似度 | 营销知识库召回 |
| TODO5 | RAG 生成 | langchain_core + langsmith | 基于知识库的营销文案 |
| TODO6 | RAGAS 评估 | numpy 规则实现 | RAG 质量评估 |

**关键收获**：
- DeepSeek V3 推理成本仅为 gpt-4o 的 ~10%，MoE 架构的革命性意义
- Prompt Engineering 是 LLM 应用的第一道工具（成本极低）
- RAG 解决 LLM 知识静态 + 无法访问私有数据的问题
- LangSmith @traceable 是 LLM 应用可观测性的标配
- RAGAS 评估是 RAG 系统持续优化的基础

**下一步**：Day 3 LLM 评估与部署--从 RAGAS 扩展到 MMLU/LLM-as-Judge 完整评估体系。
